# lists

> the things you add to over a week, that a scheduled job later consumes

In [ ]:
#| default_exp lists

In [ ]:
#| hide
from nbdev.showdoc import *
from fastcore.test import *

A list is not work and it is not a reminder. It is a pile you add to from wherever you happen to
be: a photo of yoghurt on Tuesday, three interests on Thursday. Something scheduled reads the
pile later.

An item stays open until a job says it is done. `take` is deliberately absent: a job that claims
the milk and then fails would lose the milk. So `items` reads, the work happens, and `used` marks
what actually got done.

In [ ]:
#| export
from fastcore.all import AttrDict, L, ifnone, patch, xdumps
from pobblebonk.core import Store, clip, mkid, now, row

In [ ]:
#| export
MAX_ITEM = 2_000     # chars of one item's text; a shopping line, not a document

In [ ]:
#| export
class Lists(Store):
    "Open items by topic: a shopping list, your interests, anything a later job reads."
    name = 'pob_items'
    cols = dict(id=str, topic=str, text=str, meta=str, key=str, at=float,
                used_at=float, used_by=str)
    idx = ('CREATE INDEX IF NOT EXISTS pob_items_open ON pob_items(topic, at) '
           'WHERE used_at IS NULL',
           # the same photo sent twice is one item, until something uses it
           'CREATE UNIQUE INDEX IF NOT EXISTS pob_items_key ON pob_items(topic, key) '
           'WHERE key IS NOT NULL AND used_at IS NULL')

In [ ]:
#| export
@patch
def add(self:Lists,
        topic:str,          # which list
        text:str,           # the line a model will read
        meta:dict=None,     # anything else worth keeping: a file path, a url, a photo id
        key:str=None,       # send the same thing twice and it stays one item
        at:float=None
) -> AttrDict:
    'Add one item, or give back the open one that `key` already names.'
    at = ifnone(at, now())
    r = dict(id=mkid(), topic=str(topic), text=clip(text, MAX_ITEM), meta=xdumps(meta),
             key=key, at=at, used_at=None, used_by=None)
    with self.txn():
        if key is not None and (was := self.one(
                'SELECT * FROM pob_items WHERE topic=? AND key=? AND used_at IS NULL',
                [topic, key])): return was
        self.t.insert(r)
    return row(r)

@patch
def items(self:Lists, topic:str, limit:int=200) -> L:
    'The open items on `topic`, oldest first.'
    return self.q('SELECT * FROM pob_items WHERE topic=? AND used_at IS NULL ORDER BY at LIMIT ?',
                  [str(topic), limit])

@patch
def count(self:Lists, topic:str) -> int:
    'How many items are open on `topic`. This is what a schedule with `needs` is checked against.'
    return self.one('SELECT count(*) n FROM pob_items WHERE topic=? AND used_at IS NULL',
                    [str(topic)])['n']

@patch
def used(self:Lists, ids, by:str='', at:float=None) -> int:
    'Mark items done, once the work that consumed them succeeded.'
    ids = L(ids)
    if not ids: return 0
    at, qs = ifnone(at, now()), ','.join('?'*len(ids))
    with self.txn():
        got = self.db.q(f'UPDATE pob_items SET used_at=?, used_by=? WHERE id IN ({qs}) '
                        'AND used_at IS NULL RETURNING id', [at, str(by), *ids])
    return len(got)

@patch
def drop(self:Lists, item_id:str) -> bool:
    'Remove one item outright, for something added by mistake.'
    with self.txn():
        return bool(self.db.q('DELETE FROM pob_items WHERE id=? RETURNING id', [item_id]))

@patch
def topics(self:Lists) -> L:
    'Every topic with something open on it, and how much.'
    return self.q('SELECT topic, count(*) n, min(at) since FROM pob_items WHERE used_at IS NULL '
                  'GROUP BY topic ORDER BY topic')

@patch
def history(self:Lists, topic:str, limit:int=200) -> L:
    'Items already used, newest first: what went in the cart last week.'
    return self.q('SELECT * FROM pob_items WHERE topic=? AND used_at IS NOT NULL '
                  'ORDER BY used_at DESC LIMIT ?', [str(topic), limit])

A week of shopping, then Wednesday night:

In [ ]:
from pobblebonk.core import vault

ls = Lists(vault())
ls.add('shopping', 'yoghurt, the greek one', meta=dict(photo='img_2201.heic'))
ls.add('shopping', 'oat milk')
ls.add('interests', 'sanskrit grammar')

test_eq(ls.count('shopping'), 2)
test_eq(ls.items('shopping').attrgot('text'), ['yoghurt, the greek one', 'oat milk'])
test_eq(ls.items('shopping')[0].meta, {'photo': 'img_2201.heic'})
test_eq(ls.topics().attrgot('topic'), ['interests', 'shopping'])

The same photo sent twice is one item:

In [ ]:
a = ls.add('shopping', 'yoghurt again', key='img_2201.heic')
b = ls.add('shopping', 'yoghurt again', key='img_2201.heic')
test_eq(a.id, b.id)
test_eq(ls.count('shopping'), 3)

Reading the list does not empty it. Only the job that succeeded does that, which is why a failed
cart run does not lose your milk:

In [ ]:
got = ls.items('shopping')
test_eq(ls.count('shopping'), 3)                  # still all there after reading
test_eq(ls.used(got.attrgot('id'), by='cart'), 3)
test_eq(ls.count('shopping'), 0)
test_eq(ls.used(got.attrgot('id'), by='cart'), 0)  # and marking twice does nothing the second time
test_eq(ls.history('shopping').attrgot('used_by'), ['cart']*3)

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()